<a href="https://colab.research.google.com/github/MuzaffarIshmurotov/FLYRANK/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**My lane:** Lane 2 — Refresh / Content Opportunity Scoring.

**Task type (short answer):** *Supervised binary classification, used as ranking.*

Both halves of that phrase matter, and they describe two different moments in the same pipeline:

- **At training time — supervised binary classification.** Each page carries a binary label (declining vs not-declining). The model learns a function that maps the page's features (position, freshness, engagement, content type, etc.) to a predicted probability that the page belongs to the "declining" class. Standard classifiers apply: logistic regression, decision trees, random forests, gradient boosting.

- **At inference / decision time — ranking (top-K).** The reviewer never sees the raw probability. They see a **ranked queue of the top K pages** sorted by that probability (K set by weekly reviewer capacity — 20 or 50). What matters operationally is *which pages sit at the top of the queue*, not whether the model outputs 0.71 vs 0.73 for any given page.

**Why not just call it "classification"?** Because a purely classification framing would report accuracy or F1 on the full dataset, which does not match how the output is used. The reviewer does not act on the whole dataset — they act on the top K. Naming the ranking usage forces the metric to match the decision (Section 3).

**Why not just call it "ranking"?** Because at training time, the loss function is binary cross-entropy on labeled examples — that is classification. Ignoring the classification side would obscure that the model needs a proper target (Section 2), a proper split (client-holdout), and standard classification diagnostics as secondary metrics.

**Mapping to the framing skill's task-type table.** From `framing-ml-problems/SKILL.md`, my question sounds like *"Which ones first?"* — which the table maps to *Ranking / scoring* with *precision@K* as the typical metric. This matches the framing I chose, and the classification-at-training / ranking-at-inference split is how that mapping is implemented in code.

In [1]:
# Section 1 confirmation: task type recorded, machine-readable.
task = {
    "lane":                 "Lane 2 — Refresh / Content Opportunity Scoring",
    "task_type_training":   "supervised binary classification",
    "task_type_inference":  "ranking (top-K by predicted probability)",
    "primary_metric":       "Precision@K (K = 20 and 50)",
    "framing_skill_row":    "'Which ones first?' -> Ranking / scoring -> precision@K",
}
for k, v in task.items():
    print(f"  {k:<22} : {v}")

  lane                   : Lane 2 — Refresh / Content Opportunity Scoring
  task_type_training     : supervised binary classification
  task_type_inference    : ranking (top-K by predicted probability)
  primary_metric         : Precision@K (K = 20 and 50)
  framing_skill_row      : 'Which ones first?' -> Ranking / scoring -> precision@K


## 2. Target or proxy

**What I would predict:** the column `is_declining_label`, defined as `trend_direction == "down"`.

**Where does this label come from — observed outcome, or a defined rule?**

It is a **defined proxy, not an observed outcome.** The label is computed inside the starter feature pipeline from a single column (`trend_direction`), which itself is derived from `trend_pct`. This means:

- The label is a rule-based bucket on measurements from the **same trailing 90-day window** as the features.
- It is **not** a future-window observation of what actually happened next.
- Any feature that carries information about `trend_direction` or `trend_pct` would trivially leak the label — which is why `trend_direction` and `trend_pct` are never features (documented in `flyrank-data/SKILL.md` as "the label trap").

I use the phrase **proxy** deliberately. The framing skill (`framing-ml-problems/SKILL.md`) is explicit:

> *"The target must be observed, not defined. A label that comes from someone's rule means your model learns the rule, not the world. Prefer outcomes measured in a later time window."*

Under that rule, `is_declining_label` is a beginner-grade target. It is fine for Assignment 3's teaching purpose — showing the mechanics of a classification-used-as-ranking pipeline — but it is not the target a strong capstone would ship.

**What a stronger target would look like (planned for the capstone).** Using the warehouse release (`fact_content_daily_performance`, ~79M daily rows over 17 months), I would replace the proxy with a **future-window observed outcome**, for example:

This is a genuine past-features -> future-outcome supervised learning setup, with a leakage-safe gap between the feature window and the target window (the two golden rules from the framing skill: (1) target must be observed, not defined, and (2) name the metric before training).

**For this assignment, though:**
- Target column: `is_declining_label`
- Positive class definition: `trend_direction == "down"`
- Class 1 rate on the starter data (Assignment 2, Section 3): **~54.2%** — enough positives to learn from, not degenerate
- Off-limits as features: `trend_direction`, `trend_pct` (label leakage)
- Off-limits as features: `content_id`, `client_id` (pseudonymous IDs — grouping only, per `flyrank-data/SKILL.md`)

*Observed / directional:* I do not claim the proxy label captures "declining pages" perfectly — it captures pages the current-window rule labeled as `down`. The capstone will strengthen this.

In [2]:
# Section 2 confirmation: target/proxy commitments, machine-readable.
target = {
    "column":                    "is_declining_label",
    "definition":                'trend_direction == "down"',
    "kind":                      "proxy (defined, not observed)",
    "positive_class_rate":       "~0.542 on starter data (Assignment 2 Section 3)",
    "leakage_risk_features":     ["trend_direction", "trend_pct"],
    "grouping_only_columns":     ["content_id", "client_id"],
    "capstone_upgrade_planned":  "future-window observed decline over next 30 days, warehouse release",
}
for k, v in target.items():
    print(f"  {k:<28} : {v}")

  column                       : is_declining_label
  definition                   : trend_direction == "down"
  kind                         : proxy (defined, not observed)
  positive_class_rate          : ~0.542 on starter data (Assignment 2 Section 3)
  leakage_risk_features        : ['trend_direction', 'trend_pct']
  grouping_only_columns        : ['content_id', 'client_id']
  capstone_upgrade_planned     : future-window observed decline over next 30 days, warehouse release


## 3. Success metric

**Primary metric — Precision@K, with K = 20 and K = 50.**

Precision@K is the fraction of the model's top-K ranked pages that are truly positive (declining). If the model's top 50 pages contain 37 that are actually declining, Precision@50 = 0.74.

$$
\text{Precision@}K \;=\; \frac{1}{K} \sum_{i \in \text{TopK}} y_i
$$

where the sum runs over the K pages with the highest predicted scores.

**Why Precision@K, and not accuracy or plain precision.**

- **Not accuracy.** The starter data is roughly 54% declining. A model that predicts "declining" for every page scores 54% accuracy and delivers a completely useless queue — no prioritization, no signal, just the base rate. Accuracy hides that failure. Precision@K exposes it, because a random ranking on a 54% base rate yields Precision@50 ≈ 0.54, and any real model must beat that.
- **Not plain precision (without K).** Plain precision would count every page the model flags at some threshold. But the reviewer never acts on "every page flagged." The reviewer acts on a fixed number of top-ranked pages per week (their capacity). The metric has to match that reality.

**Why K = 20 and K = 50 specifically.** K is a *business* choice, not a math choice. The framing skill's rule "name the metric before training" really means: name K first, based on the real workflow. Assignment 2's framing set:
- **K = 20** — matches a small weekly review capacity (one writer, ~20 pages to inspect deeply)
- **K = 50** — matches a larger weekly slot (a team, ~50 pages triaged)

Reporting both K values makes the model's behavior visible across two realistic reviewer capacities, not just one.

**Numerical target for "good."**

From Assignment 1's client-holdout run, the reference random forest already reached Precision@50 = 0.740 (37 of 50 top-ranked pages truly declining) versus a hand-written rule at 0.240 (12 of 50). That baseline-vs-model gap is my anchor for what "good" means on this data:

- **Baseline to beat:** Precision@50 = 0.240 (hand-written rule from Assignment 1)
- **Reference result to match or exceed with proper leakage discipline:** Precision@50 = 0.740 (random forest, client-holdout)
- **"Good" for the capstone (target):** Precision@50 ≥ 0.70 on client-holdout with a stronger, leakage-safe label, plus stable behavior across K = 20 and K = 50

**Secondary metrics (context, not the headline).**

- **ROC-AUC** — how well the model separates the two classes across the full ranking, not just the top-K. Useful diagnostic; not the metric that matches the decision.
- **Average precision (AP)** — summarizes the whole precision-recall curve. Useful when the top-K cutoff might change, but again not the metric the reviewer optimizes for.
- **Calibration** — does a predicted probability of 0.70 mean roughly 70% of such pages truly decline? Matters for interpretability and reason-code trust.
- **Per-client Precision@K variance** — because the model is validated with client-holdout, a headline Precision@50 hides whether the model is uniformly good or good on average and terrible on some clients. Reporting the distribution across held-out clients is honest.

*Observed / decision-support:* the metric is chosen to match the top-K review workflow. It does not measure "will this page recover if refreshed" — that would require a causal design.

In [3]:
# Section 3 confirmation: metric commitments, machine-readable.
# Also computes a quick sanity anchor from the Assignment 1 baseline vs random forest.

metric = {
    "primary_metric":        "Precision@K",
    "K_values":              [20, 50],
    "K_rationale":           "reviewer weekly capacity (business constraint, not tuned)",
    "not_used_and_why":      {
        "accuracy":         "misleading on ~54% positive base rate",
        "plain_precision":  "ignores fixed reviewer capacity (K)",
    },
    "secondary_metrics":     ["ROC-AUC", "average_precision", "calibration",
                              "per-client Precision@K variance"],
}

# Anchor numbers from Assignment 1 (client-holdout)
baseline_p50   = 0.240
best_model_p50 = 0.740
lift           = best_model_p50 / baseline_p50
capstone_target = 0.70

print("Metric commitments")
print("-" * 60)
for k, v in metric.items():
    print(f"  {k:<24} : {v}")

print()
print("Anchor numbers from Assignment 1 (client-holdout)")
print("-" * 60)
print(f"  Baseline Precision@50   : {baseline_p50:.3f}   (~{int(baseline_p50*50)} of top 50 correct)")
print(f"  Random forest Precision@50: {best_model_p50:.3f}   (~{int(best_model_p50*50)} of top 50 correct)")
print(f"  Relative lift             : {lift:.2f}x")
print(f"  Capstone target (>=)      : {capstone_target:.3f}")

Metric commitments
------------------------------------------------------------
  primary_metric           : Precision@K
  K_values                 : [20, 50]
  K_rationale              : reviewer weekly capacity (business constraint, not tuned)
  not_used_and_why         : {'accuracy': 'misleading on ~54% positive base rate', 'plain_precision': 'ignores fixed reviewer capacity (K)'}
  secondary_metrics        : ['ROC-AUC', 'average_precision', 'calibration', 'per-client Precision@K variance']

Anchor numbers from Assignment 1 (client-holdout)
------------------------------------------------------------
  Baseline Precision@50   : 0.240   (~12 of top 50 correct)
  Random forest Precision@50: 0.740   (~37 of top 50 correct)
  Relative lift             : 3.08x
  Capstone target (>=)      : 0.700


## 4. The unit of analysis, as a real dataframe

**Unit of analysis:** one row = one **page** (one `content_id`) as of the trailing-90-day snapshot.

Every observation the model sees, and every prediction it produces, is scoped to a single page. The reviewer's action is also per-page (open this URL, refresh or dismiss). So the modeling grain, the label grain, and the decision grain all align. This is the simplest honest grain for Lane 2 on the starter dataset — no time-series expansion, no per-keyword expansion, no aggregation to client level.

**What "the slice" means for Lane 2.** Lane 2's declared filters (from the starter pipeline and `flyrank-data/SKILL.md`) are:
- `impressions_90d > 0` — the page actually saw traffic in the 90-day window
- `content_age_days >= 90` — the page has enough history for the 90-day metrics to be meaningful
- Deduplicated by `content_id`

The code cell below applies those filters, prints the shape of the resulting dataframe, shows a small sample of rows, and sketches what the target column and a few candidate features would look like at training time.

**A note on data hygiene, from `flyrank-data/SKILL.md`:**
- Rate columns are ×100 percentages (e.g., `ctr = 0.76` means 0.76%, not 76%). The code cell prints raw values without silently rescaling — that will happen at feature-engineering time, not here.
- `avg_position = 0` means "no data," not rank zero (~1,205 rows). I do not filter those out in this framing snapshot, but I flag their count so the presence is visible.
- No client names, URLs, raw queries, or private strings are printed anywhere — only pseudonymized IDs and observable measurements.

**Observed:** the Lane 2 slice filters (`impressions_90d > 0`, `content_age_days >= 90`) removed 0 rows on this starter CSV — the file appears to have been pre-filtered to those criteria at anonymization time. This matches the same-zero-rows behavior observed in Assignment 1's Notebook 01. Not a bug; a fact about the release. On the larger warehouse dataset those filters will remove real rows.

In [6]:
# Ensure the starter repo (with the CSV) is available in the Colab session.
# Idempotent — safe to run multiple times.
import os
STARTER_PATH = "/content/flyrank-ml-internship-starter"

if not os.path.exists(STARTER_PATH):
    !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git {STARTER_PATH}
else:
    print(f"Starter repo already present at {STARTER_PATH}")

Cloning into '/content/flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 283 (delta 106), reused 78 (delta 78), pack-reused 147 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 13.86 MiB/s, done.
Resolving deltas: 100% (152/152), done.


In [7]:
# Section 4 — the unit of analysis, made real.
# Loads the starter dataset, applies Lane 2's slice filters,
# and shows: shape, a small row sample, target column preview,
# and a candidate-feature / off-limits column breakdown.

import pandas as pd

# --- 1. Load the starter dataset (Colab path) --------------------------
CANDIDATE_PATHS = [
    "/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
]

df_raw = None
for path in CANDIDATE_PATHS:
    try:
        df_raw = pd.read_csv(path)
        print(f"Loaded from: {path}")
        break
    except FileNotFoundError:
        continue

if df_raw is None:
    raise FileNotFoundError(
        "Starter CSV not found. In Colab, first run in a new cell:\n"
        "  !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git"
    )

print(f"Raw shape:                {df_raw.shape[0]:>7,} rows x {df_raw.shape[1]} columns")

# --- 2. Flag data hygiene (transparency, not silent fixes) ------------
n_no_pos = int((df_raw["avg_position"] == 0).sum()) if "avg_position" in df_raw.columns else 0
print(f"Rows with avg_position=0: {n_no_pos:>7,}   (means 'no data', not rank zero)")

# --- 3. Apply Lane 2's slice filters ----------------------------------
mask = (df_raw["impressions_90d"] > 0) & (df_raw["content_age_days"] >= 90)
df = df_raw.loc[mask].drop_duplicates(subset=["content_id"]).copy()
print(f"After Lane 2 filters:     {df.shape[0]:>7,} rows x {df.shape[1]} columns")
print(f"Removed by filters:       {df_raw.shape[0] - df.shape[0]:>7,} rows")

# --- 4. Derive the target column (proxy) ------------------------------
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# --- 5. Confirm "one row = one page" ---------------------------------
n_unique_content = df["content_id"].nunique()
assert n_unique_content == len(df), "Grain check failed: duplicated content_id present"
print(f"\nGrain check: one row per content_id? {n_unique_content == len(df)}")
print(f"  unique content_id: {n_unique_content:,} ; rows: {len(df):,}")

# --- 6. Small row sample (a few informative columns only) -------------
sample_cols = [
    "content_id", "client_id",
    "content_type", "main_intent",
    "impressions_90d", "avg_position", "ctr",
    "content_age_days", "days_since_last_update",
    "trend_direction", "is_declining_label",
]
sample_cols = [c for c in sample_cols if c in df.columns]

print("\nSample rows (first 5):")
print("-" * 70)
print(df[sample_cols].head().to_string(index=False))

# --- 7. Target column preview ----------------------------------------
label_counts = df["is_declining_label"].value_counts().sort_index()
label_rate   = df["is_declining_label"].mean()
print("\nTarget column preview")
print("-" * 70)
print(f"  is_declining_label = 1 (declining):    {label_counts.get(1, 0):>6,}")
print(f"  is_declining_label = 0 (not declining):{label_counts.get(0, 0):>6,}")
print(f"  Positive-class rate:                   {label_rate:.3f}")

# --- 8. Column role map -----------------------------------------------
label_col      = ["is_declining_label"]
leakage_cols   = ["trend_direction", "trend_pct"]        # derive-the-label, NEVER features
id_cols        = ["content_id", "client_id"]              # grouping / splits only
candidate_features = [c for c in df.columns
                      if c not in label_col + leakage_cols + id_cols]

print("\nColumn role map")
print("-" * 70)
print(f"  Target column        : {label_col}")
print(f"  Off-limits (leakage) : {leakage_cols}")
print(f"  Off-limits (ID only) : {id_cols}")
print(f"  Candidate features   : {len(candidate_features)} columns  (subset shown)")
print(f"    e.g., {candidate_features[:8]} ...")

Loaded from: /content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv
Raw shape:                 30,000 rows x 44 columns
Rows with avg_position=0:   1,205   (means 'no data', not rank zero)
After Lane 2 filters:      30,000 rows x 44 columns
Removed by filters:             0 rows

Grain check: one row per content_id? True
  unique content_id: 30,000 ; rows: 30,000

Sample rows (first 5):
----------------------------------------------------------------------
          content_id         client_id    content_type   main_intent  impressions_90d  avg_position  ctr  content_age_days  days_since_last_update trend_direction  is_declining_label
content_304f48230142 client_f369cb89fc keyword article transactional             3803          10.6 0.76               187                      20            down                   1
content_a1fb4e703a9e client_4e07408562 keyword article informational            15320          20.3 0.05               445                      25   

## 5. Why ML beats a fixed rule here

Assignment 2 argued the *business* case for ML: reviewer capacity is fixed, so raw accuracy is beside the point, and what matters is *ranking under capacity*. This section makes the technical case for why a fixed if-statement can't match a learned ranker on this specific data.

### 1. The signal is joint, not marginal

Decline is not "high position + low CTR" alone, nor "old page + low freshness" alone. It's the **joint pattern** of position × freshness × content_type × engagement_rate × query mix that carries the signal. A hand-written rule made of AND/OR clauses over individual thresholds captures marginal effects; it misses interactions. Tree-based models (decision tree, random forest, gradient boosting) capture interactions natively by splitting on features conditional on other features already having been split — which is exactly the shape of the signal.

### 2. The right thresholds vary across subgroups

A page at position 3 with a CTR of 5% is a red flag; the same 5% CTR at position 30 is expected. A single hand-written threshold like `ctr < 0.10` will over-flag long-tail pages and under-flag first-page pages. The reference pipeline's use of `position_tier` (`page_1`, `top_3`, `striking`, `page_3_5`, `deep`) helps somewhat, but a hand rule then needs a different threshold per tier and per content_type — the rule table explodes combinatorially. A learned model estimates these subgroup-specific thresholds implicitly from data.

### 3. The signal is spread across many weakly-informative features

The starter feature set includes ~40 candidate features (Section 4). No single feature has a strong marginal correlation with the label (Assignment 1's Discovery A showed near-zero correlation between `search_volume` and `impressions_90d`). Instead, each feature carries a small amount of information. A hand-written rule that picks one or two "obvious" features throws away most of the available signal. A learned model integrates weak signals across many features — which is exactly where ensembles like random forest have historically dominated tabular ranking.

### 4. Ranking under fixed capacity is the exact regime where ML helps most

The reviewer sees the top K, not the whole distribution. This makes **ordering quality among the highest-risk pages** the thing that matters. Hand-written scoring rules can produce a ranking, but the ordering is coarse (ties are common; scoring is often multi-step arithmetic). A model that outputs a smooth continuous probability produces a fine-grained ranking that discriminates well right at the cutoff. Small ordering improvements near rank K translate directly into Precision@K gains.

### 5. Empirical evidence, on this exact dataset

The strongest argument is not theoretical, it's measured. Assignment 1's reference pipeline reports, on client-holdout validation:

| Method                    | Precision@50 | Top-50 correct |
|---------------------------|:------------:|:--------------:|
| Hand-written baseline     | **0.240**    | ~12 / 50       |
| Logistic regression       | 0.400        | ~20 / 50       |
| Decision tree             | 0.540        | ~27 / 50       |
| **Random forest**         | **0.740**    | **~37 / 50**   |

A 3.08× lift over the transparent rule on the same holdout is not a projection — it's an already-observed result on this exact data. The rule is not just a little worse; it's meaningfully worse at the metric that matches the decision.

### What ML would NOT beat a rule at

To be honest about limits — a rule wins in three cases, none of which describes Lane 2:

- **When the signal is a single, stable threshold** (e.g., "flag pages that returned 404 this week"). A rule is simpler and equally accurate.
- **When the label is fully derivable from one column** — but then no model is needed at all.
- **When explainability requirements are so strict that a stakeholder demands a transparent audit trail per decision.** Even here, the modern answer is *baseline blended with model plus reason codes*, exactly what the reference pipeline does in Stage 4.

*Observed / decision-support:* ML earns its place on Lane 2 because the pattern is joint, threshold-varying, spread across many weak features, and evaluated under a top-K constraint — the exact regime where ensembles dominate rules empirically on tabular data.

In [8]:
# Section 5 confirmation: technical arguments for ML > rule, machine-readable.
# Includes the Assignment 1 empirical anchor.

rule_vs_ml = {
    "why_ml_wins_here": [
        "signal is joint across features, not marginal",
        "correct thresholds vary across position_tier x content_type",
        "signal is spread across many weakly-informative features",
        "top-K ranking under fixed reviewer capacity is the ensemble sweet spot",
    ],
    "when_a_rule_would_win_instead": [
        "single stable threshold captures the whole signal",
        "label fully derivable from one column (no model needed)",
        "hard explainability constraints (mitigated by baseline + model + reason codes)",
    ],
}

# Empirical anchor from Assignment 1 (client-holdout Precision@50)
results_p50 = {
    "hand_written_baseline": 0.240,
    "logistic_regression":   0.400,
    "decision_tree":         0.540,
    "random_forest":         0.740,
}

print("Why ML > rule for Lane 2")
print("-" * 60)
for k, v in rule_vs_ml.items():
    print(f"  {k}:")
    for item in v:
        print(f"    - {item}")

print()
print("Empirical anchor (Assignment 1, client-holdout, Precision@50)")
print("-" * 60)
baseline = results_p50["hand_written_baseline"]
for method, p50 in results_p50.items():
    lift = p50 / baseline
    marker = "  <-- best" if method == "random_forest" else ""
    print(f"  {method:<24} : {p50:.3f}   (lift over baseline: {lift:.2f}x){marker}")

Why ML > rule for Lane 2
------------------------------------------------------------
  why_ml_wins_here:
    - signal is joint across features, not marginal
    - correct thresholds vary across position_tier x content_type
    - signal is spread across many weakly-informative features
    - top-K ranking under fixed reviewer capacity is the ensemble sweet spot
  when_a_rule_would_win_instead:
    - single stable threshold captures the whole signal
    - label fully derivable from one column (no model needed)
    - hard explainability constraints (mitigated by baseline + model + reason codes)

Empirical anchor (Assignment 1, client-holdout, Precision@50)
------------------------------------------------------------
  hand_written_baseline    : 0.240   (lift over baseline: 1.00x)
  logistic_regression      : 0.400   (lift over baseline: 1.67x)
  decision_tree            : 0.540   (lift over baseline: 2.25x)
  random_forest            : 0.740   (lift over baseline: 3.08x)  <-- best


## Self-check

Before I submit, I confirm each line honestly:

- [x] **Every section above is filled** — markdown thinking AND the code that backs it
  (Section 1: task type. Section 2: target/proxy. Section 3: success metric. Section 4: unit of analysis with a real dataframe. Section 5: why ML > rule.)
- [x] **The notebook runs top to bottom with no errors** (Runtime → Run all — verified; clone cell at the top ensures the starter CSV is available)
- [x] **No client names, URLs, or private queries anywhere** — all IDs pseudonymized (`content_id`, `client_id`); no raw fields printed
- [x] **My claims use careful words** — observed, measured, directional, decision-support; no causal or algorithmic claims
- [x] **Committed to my repo under `work/notebooks/`** — file at `work/notebooks/w02_ml_task_framing.ipynb`, then submit repo URL on the InternHQ card

### Pass-bar cross-check (from the assignment card)

- [x] **Names the ML task type** → Section 1: *supervised binary classification, used as ranking (top-K by predicted probability)*
- [x] **Names the target/proxy** → Section 2: `is_declining_label` as a **defined proxy** (not observed); capstone upgrade to future-window observed outcome planned
- [x] **Names the success metric** → Section 3: **Precision@K** (K = 20 and 50), with baseline anchor (0.240) and capstone target (≥ 0.70)
- [x] **Shows the unit of analysis as a real dataframe** → Section 4: loaded, filtered, and printed — one row = one `content_id`, grain assertion passes (30,000 unique / 30,000 rows), positive-class rate 0.542
- [x] **Explains why this is an ML/analysis problem, not just a rule** → Section 5: four technical reasons (joint signal, subgroup-varying thresholds, weak signal spread across ~40 features, top-K regime) plus empirical anchor (0.240 → 0.740 = 3.08× on the same holdout)
- [x] **Ties the output to a real content action** → Section 1 + 3: the ranked queue of top-K pages goes to a content strategist / SEO writer whose weekly capacity defines K; the metric Precision@K is chosen to match that decision

### Provisional commitment (from Assignment 2)

Lane 2 (Refresh / Content Opportunity Scoring) remains my provisional lane. Assignment 3 sharpens the technical framing; it does not commit me to a lane change. I retain the right to change lane until end of Week 4 per program guidance.

### Language discipline reminder

Throughout the capstone I will phrase outputs as *estimates*, *risk scores*, or *ranked review candidates*, never as *predictions of future traffic*, *proof of decline*, or *guaranteed recovery targets*. This is a decision-support system, not an autonomous action system.